DOSPERT Analysis Notebook


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.api as sm
from pathlib import Path


In [ ]:
# LOAD DATA 


#two separate file paths
FMRI_DATA_PATH = "/scratch/snb3/fmri_project/DOSPERT_subject_domain.csv"       # sub-001, sub-002, sub-003  (DOSPERT + fMRI)
POP_DATA_PATH  = "/scratch/snb3/fmri_project/Pop_Organized_Domain.csv" # sub-004 … sub-013          (DOSPERT only)

OUTPUT_DIR = Path("dospert_results")
OUTPUT_DIR.mkdir(exist_ok=True)

# Group labels dataframe
GROUP_FMRI = "fMRI"
GROUP_POP  = "Population"

# Column names
COL_MAP = {
    "participant": "Participant ID",
    "item":        "Item",
    "rt":          "RT",
    "eb":          "EB",
    "pr":          "PR",
    "domain":      "Domain",
    "age":         "Age",
    "gender":      "Gender",
}

DOMAINS = ["Ethical", "Financial", "Health/Safety", "Recreational", "Social"]

In [ ]:
# STEP 1: LOAD DATA

#HELPER FUNCTIONS

def _load_single_csv(path: str, group_label: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    required = list(COL_MAP.values())
    missing  = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"[{path}] Missing columns: {missing}\n"
            f"  Found: {list(df.columns)}"
        )

    
    inv_map = {v: k for k, v in COL_MAP.items()}
    df = df.rename(columns=inv_map)

    # Label group
    df["group"] = group_label

    # numeric scores
    for col in ["rt", "eb", "pr"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    n_dropped = df[["rt", "eb", "pr"]].isna().any(axis=1).sum()
    if n_dropped:
        print(f"  ⚠  [{path}] Dropped {n_dropped} rows with non-numeric RT/EB/PR.")
        df = df.dropna(subset=["rt", "eb", "pr"])

    print(f"  ✓  [{group_label}]  {path}")
    print(f"     {len(df)} rows | {df['participant'].nunique()} participants: "
          f"{sorted(df['participant'].unique().tolist())}")
    return df


def load_data(fmri_path: str, pop_path: str) -> pd.DataFrame:
    """Load both CSVs and return a single concatenated DataFrame."""
    print(f"\n  Loading fMRI subjects from  : {fmri_path}")
    df_fmri = _load_single_csv(fmri_path, GROUP_FMRI)

    print(f"\n  Loading population subjects from : {pop_path}")
    df_pop  = _load_single_csv(pop_path,  GROUP_POP)

    # Check for overlapping participant IDs 
    overlap = set(df_fmri["participant"]) & set(df_pop["participant"])
    if overlap:
        print(f"\n  ⚠  WARNING: Participant IDs appear in BOTH files: {overlap}")
        print(     "     Verify your data — each participant should be in exactly one file.")

    df = pd.concat([df_fmri, df_pop], ignore_index=True)
    print(f"\n  ── Combined dataset ──")
    print(f"     {len(df)} total rows | "
          f"{df['participant'].nunique()} participants | "
          f"Groups: {df.groupby('group')['participant'].nunique().to_dict()}")
    return df




In [ ]:

#STEP 2: Per participant regression
#     RT  =  a*EB  +  b*PR  +  c
#a = Expected benefit sensitivity
# =b Risk Perception sensitivity
# risk_attitude is the mean predicted RT

def fit_participant_regressions(df: pd.DataFrame) -> pd.DataFrame:
   
    records = []
    for pid, sub in df.groupby("participant"):
        if len(sub) < 3:
            print(f"  ⚠  {pid}: only {len(sub)} items – skipping regression.")
            continue

        X = sm.add_constant(sub[["eb", "pr"]])
        y = sub["rt"]
        try:
            model  = sm.OLS(y, X).fit()
            a      = model.params.get("eb", np.nan)
            b      = model.params.get("pr", np.nan)
            c      = model.params.get("const", np.nan)
            a_p    = model.pvalues.get("eb", np.nan)
            b_p    = model.pvalues.get("pr", np.nan)
            r2     = model.rsquared
            # Risk attitude: mean predicted preference
            risk_attitude = model.fittedvalues.mean()
        except Exception as e:
            print(f"  ⚠  {pid}: regression failed ({e})")
            a = b = c = a_p = b_p = r2 = risk_attitude = np.nan

        row_meta = sub.iloc[0]
        records.append({
            "participant":  pid,
            "group":        row_meta["group"],
            "age":          row_meta["age"],
            "gender":       row_meta["gender"],
            "a":            a,        # Expected-Benefit coefficient
            "b":            b,        # Risk-Perception coefficient
            "intercept":    c,
            "a_pval":       a_p,
            "b_pval":       b_p,
            "r_squared":    r2,
            "risk_attitude": risk_attitude,
            "risk_label":   "risk-seeking" if (b is not np.nan and b > 0) else "risk-averse",
        })

    coef_df = pd.DataFrame(records)
    print(f"\n── Regression coefficients ({len(coef_df)} participants) ──")
    print(coef_df[["participant","group","a","b","r_squared","risk_label"]].to_string(index=False))
    return coef_df



In [ ]:

#STEP 3: Group ANOVA / GLM


def run_anova(coef_df: pd.DataFrame) -> dict:
    results = {}

    for dep_var, label in [("a", "EB coefficient (a)"),
                            ("b", "PR coefficient (b)"),
                            ("risk_attitude", "Risk Attitude")]:
        clean = coef_df.dropna(subset=[dep_var, "group"])
        if clean["group"].nunique() < 2:
            print(f"  ⚠  Not enough groups for ANOVA on {dep_var}.")
            continue

        # One-way ANOVA
        groups  = [g[dep_var].values for _, g in clean.groupby("group")]
        f_stat, p_val = stats.f_oneway(*groups)

        # GLM with age and gender as covariates 
        formula = f"{dep_var} ~ C(group)"
        if clean["age"].notna().sum() > len(clean) * 0.5:
            formula += " + age"
        if clean["gender"].notna().sum() > len(clean) * 0.5:
            formula += " + C(gender)"

        try:
            glm_model  = ols(formula, data=clean).fit()
            glm_table  = anova_lm(glm_model, typ=2)
        except Exception as e:
            glm_table = None
            print(f"  ⚠  GLM failed for {dep_var}: {e}")

        # stats by group
        desc = clean.groupby("group")[dep_var].agg(["mean","std","count"])

        results[dep_var] = {
            "label":     label,
            "f_stat":    f_stat,
            "p_val":     p_val,
            "desc":      desc,
            "glm_table": glm_table,
        }

        print(f"\n── ANOVA: {label} ──")
        print(desc.round(3).to_string())
        print(f"  F = {f_stat:.3f},  p = {p_val:.4f}  {'*' if p_val < .05 else '(ns)'}")
        if glm_table is not None:
            print("\n  GLM (Type II SS):")
            print(glm_table.round(4).to_string())

    return results


In [ ]:
#STEP 4: BY DOMAIN-LEVEL 

def domain_analysis(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for (pid, dom), sub in df.groupby(["participant", "domain"]):
        if len(sub) < 3:
            continue
        X = sm.add_constant(sub[["eb", "pr"]])
        y = sub["rt"]
        try:
            model = sm.OLS(y, X).fit()
            a = model.params.get("eb", np.nan)
            b = model.params.get("pr", np.nan)
        except Exception:
            a = b = np.nan
        row_meta = sub.iloc[0]
        records.append({
            "participant": pid,
            "group":       row_meta["group"],
            "domain":      dom,
            "a_domain":    a,
            "b_domain":    b,
        })
    return pd.DataFrame(records)



In [ ]:
#STEP 5: VISUALIZE

palette_colors = {"fMRI": "#E05A5A", "Population": "#4A90D9"}

def plot_coefficient_distributions(coef_df, anova_results, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    fig.suptitle("DOSPERT Regression Coefficients by Group", fontsize=14, fontweight="bold")

    for ax, (dep_var, label) in zip(axes, [
            ("a",            "EB coefficient (a)"),
            ("b",            "PR coefficient (b)"),
            ("risk_attitude", "Risk Attitude (mean predicted RT)")]):

        clean = coef_df.dropna(subset=[dep_var])
        for grp, sub in clean.groupby("group"):
            ax.hist(sub[dep_var], bins=8, alpha=0.65,
                    label=grp, color=palette_colors.get(grp, "grey"), edgecolor="white")
            ax.axvline(sub[dep_var].mean(), color=palette_colors.get(grp, "grey"),
                       lw=2, linestyle="--")

        ax.set_title(label, fontsize=11)
        ax.set_xlabel("Coefficient value")
        ax.set_ylabel("Count")
        ax.legend(fontsize=9)
        if dep_var in anova_results:
            p = anova_results[dep_var]["p_val"]
            ax.set_title(f"{label}\n(F-test p = {p:.3f}{'*' if p < .05 else ''})", fontsize=10)

        # Mark 0 for b (risk-seeking vs averse boundary)
        if dep_var == "b":
            ax.axvline(0, color="black", lw=1, linestyle=":")

    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  ✓  Saved: {save_path}")


def plot_scatter_ab(coef_df, save_path):
    fig, ax = plt.subplots(figsize=(7, 5))
    for grp, sub in coef_df.dropna(subset=["a","b"]).groupby("group"):
        ax.scatter(sub["a"], sub["b"], label=grp,
                   color=palette_colors.get(grp, "grey"), s=80, alpha=0.85, edgecolors="white")
        # annotate participant IDs
        for _, row in sub.iterrows():
            ax.annotate(str(row["participant"]), (row["a"], row["b"]),
                        fontsize=7, alpha=0.7,
                        xytext=(4, 4), textcoords="offset points")

    ax.axhline(0, color="black", lw=1, linestyle="--", alpha=0.5)
    ax.axvline(0, color="black", lw=1, linestyle="--", alpha=0.5)
    ax.set_xlabel("a  (Expected Benefit coefficient)")
    ax.set_ylabel("b  (Risk Perception coefficient)\n← risk-averse  |  risk-seeking →")
    ax.set_title("Participant Risk Profiles  (a vs b)")
    ax.legend()
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  ✓  Saved: {save_path}")


def plot_domain_heatmap(domain_df, save_path):
    if domain_df.empty:
        return
    pivot = domain_df.pivot_table(index="participant", columns="domain",
                                   values="b_domain", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns)*1.4), max(4, len(pivot)*0.6)))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
                linewidths=0.4, ax=ax)
    ax.set_title("PR coefficient (b) by Participant × Domain\n(blue = risk-averse, red = risk-seeking)")
    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  ✓  Saved: {save_path}")


def plot_group_means(anova_results, save_path):
    fig, axes = plt.subplots(1, len(anova_results), figsize=(5*len(anova_results), 5))
    if len(anova_results) == 1:
        axes = [axes]

    for ax, (dep_var, res) in zip(axes, anova_results.items()):
        desc = res["desc"]
        groups = desc.index.tolist()
        means  = desc["mean"].values
        sems   = (desc["std"] / np.sqrt(desc["count"])).values
        colors = [palette_colors.get(g, "grey") for g in groups]

        bars = ax.bar(groups, means, yerr=sems, capsize=6,
                      color=colors, alpha=0.85, edgecolor="white", width=0.5)
        ax.set_title(res["label"], fontsize=11)
        ax.set_ylabel("Mean coefficient (±SEM)")
        p = res["p_val"]
        ax.set_xlabel(f"Group  (F-test p = {p:.3f}{'*' if p < .05 else ' ns'})")

        if dep_var == "b":
            ax.axhline(0, color="black", lw=1, linestyle="--", alpha=0.5)

    plt.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  ✓  Saved: {save_path}")


In [ ]:
#STEP 6. SAVE
#files are saved as output csv files 


def save_results(coef_df, domain_df, anova_results, out_dir):
    coef_df.to_csv(out_dir / "participant_coefficients.csv", index=False)
    print(f"  ✓  participant_coefficients.csv")

    if not domain_df.empty:
        domain_df.to_csv(out_dir / "domain_coefficients.csv", index=False)
        print(f"  ✓  domain_coefficients.csv")

    lines = []
    for dep_var, res in anova_results.items():
        lines.append(f"\n=== {res['label']} ===")
        lines.append(res["desc"].round(4).to_string())
        lines.append(f"F = {res['f_stat']:.4f},  p = {res['p_val']:.4f}")
        if res["glm_table"] is not None:
            lines.append("\nGLM (Type II SS):")
            lines.append(res["glm_table"].round(4).to_string())
    with open(out_dir / "anova_results.txt", "w") as f:
        f.write("\n".join(lines))
    print(f"  ✓  anova_results.txt")


In [ ]:

#full pipeline

def main():

    # 1. Load both CSVs and combine
    print("Step 1 – Loading data …")
    df = load_data(FMRI_DATA_PATH, POP_DATA_PATH)

    # 2. Per-participant regressions
    print("\n Step 2 – Fitting per-participant regressions …")
    coef_df = fit_participant_regressions(df)

    # 3. ANOVA / GLM
    print("\n Step 3 – Running ANOVA / GLM …")
    anova_results = run_anova(coef_df)

    # 4. Domain breakdown
    print("\n Step 4 – Domain-level analysis …")
    domain_df = domain_analysis(df)

    # 5. Plots
    print("\n Step 5 – Generating plots …")
    plot_coefficient_distributions(coef_df, anova_results,
        OUTPUT_DIR / "coefficient_distributions.png")
    plot_scatter_ab(coef_df,
        OUTPUT_DIR / "scatter_a_vs_b.png")
    plot_domain_heatmap(domain_df,
        OUTPUT_DIR / "domain_heatmap.png")
    if anova_results:
        plot_group_means(anova_results,
            OUTPUT_DIR / "group_means.png")

    # 6. Save
    print("\n Step 6 – Saving results …")
    save_results(coef_df, domain_df, anova_results, OUTPUT_DIR)

    print("\n Results saved to:", OUTPUT_DIR.resolve())



if __name__ == "__main__":
    main()
